In [1]:
import pandas as pd
import numpy as np
from cmdstanpy import CmdStanModel
import geopandas as gpd
from libpysal import weights
from sklearn.preprocessing import LabelEncoder
import networkx as nx
import re
import os
import pickle
from scipy.sparse import diags

/home/yuyue/stan_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
strat_frame = pd.read_csv("/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/state_LP_cleaned_sf.csv")

In [3]:
strat_frame

,Unnamed: 0.1,Unnamed: 0,sex_idx,age_idx,eth_idx,income_idx,state_idx,vote_idx,education_idx,N_star.1,...,Belief_level_of_concern_about_the_possibility_of_ukraine_being_defeated_and_taken_over_by_russia_Symbol_LP,Belief_opinion_on_whether_the_us_has_a_responsibility_to_help_ukraine_defend_itself_from_russia_s_invasion_Symbol_LP,Belief_opinion_on_the_following_statement_it_should_be_acceptable_for_a_pregnant_woman_to_obtain_a_legal_abortion_if_a_woman_wants_it_for_any_reason_Symbol_LP,Belief_opinion_on_the_statement_a_person_should_require_a_police_permit_before_he_or_she_could_buy_a_gun_Symbol_LP,Belief_opinion_on_the_statement_the_use_of_puberty_blockers_and_other_genderaffirming_procedures_for_minors_should_be_allowed_Symbol_LP,Belief_increase_fossil_fuel_production_Symbol_LP,Belief_opinion_on_funding_of_police_departments_Symbol_LP,Belief_opinion_on_whether_white_people_benefit_from_societal_advantages_not_available_to_black_people_Symbol_LP,Belief_opinion_on_the_statement_the_us_should_provide_military_aid_to_israel_to_help_in_its_war_against_hamas_Symbol_LP_2,Belief_opinion_on_the_statement_the_us_should_provide_military_aid_to_israel_to_help_in_its_war_against_hamas_Symbol_LP
0,0,0,2,1,4,1,1,1,1,11.20473,...,0.094125,0.399268,-1.207061,-2.671429,-1.563661,0.595799,-0.422827,-0.481602,-0.487338,0.133341
1,1,1,2,1,4,1,1,2,1,0.08251,...,0.094125,0.399268,-1.207061,-2.671429,-1.563661,0.595799,-0.422827,-0.481602,-0.487338,0.133341
2,2,2,2,1,4,1,1,3,1,1.58995,...,0.094125,0.399268,-1.207061,-2.671429,-1.563661,0.595799,-0.422827,-0.481602,-0.487338,0.133341
3,3,3,2,1,4,1,1,4,1,0.01103,...,0.094125,0.399268,-1.207061,-2.671429,-1.563661,0.595799,-0.422827,-0.481602,-0.487338,0.133341
4,4,4,2,1,4,1,1,6,1,0.09892,...,0.094125,0.399268,-1.207061,-2.671429,-1.563661,0.595799,-0.422827,-0.481602,-0.487338,0.133341
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190839,190839,190839,1,6,1,5,51,3,2,183.35671,...,-1.985449,-1.998531,-1.437020,-4.230682,-1.021586,0.451635,0.910535,-1.525207,-1.526627,-2.663479
190840,190840,190840,1,6,1,5,51,4,1,0.27655,...,-1.985449,-1.998531,-1.437020,-4.230682,-1.021586,0.451635,0.910535,-1.525207,-1.526627,-2.663479
190841,190841,190841,1,6,1,5,51,4,2,0.94979,...,-1.985449,-1.998531,-1.437020,-4.230682,-1.021586,0.451635,0.910535,-1.525207,-1.526627,-2.663479
190842,190842,190842,1,6,1,5,51,6,1,0.47236,...,-1.985449,-1.998531,-1.437020,-4.230682,-1.021586,0.451635,0.910535,-1.525207,-1.526627,-2.663479


In [4]:
strat_frame['N_star.median'].sum()

np.float64(10370279.246645)

In [5]:
strat_frame.columns.tolist()

['Unnamed: 0.1',
 'Unnamed: 0',
 'sex_idx',
 'age_idx',
 'eth_idx',
 'income_idx',
 'state_idx',
 'vote_idx',
 'education_idx',
 'N_star.1',
 'N_star.2',
 'N_star.3',
 'N_star.4',
 'N_star.5',
 'N_star.6',
 'N_star.7',
 'N_star.8',
 'N_star.9',
 'N_star.10',
 'N_star.11',
 'N_star.12',
 'N_star.13',
 'N_star.14',
 'N_star.15',
 'N_star.16',
 'N_star.17',
 'N_star.18',
 'N_star.19',
 'N_star.20',
 'N_star.21',
 'N_star.22',
 'N_star.23',
 'N_star.24',
 'N_star.25',
 'N_star.26',
 'N_star.27',
 'N_star.28',
 'N_star.29',
 'N_star.30',
 'N_star.31',
 'N_star.32',
 'N_star.33',
 'N_star.34',
 'N_star.35',
 'N_star.36',
 'N_star.37',
 'N_star.38',
 'N_star.39',
 'N_star.40',
 'N_star.41',
 'N_star.42',
 'N_star.43',
 'N_star.44',
 'N_star.45',
 'N_star.46',
 'N_star.47',
 'N_star.48',
 'N_star.49',
 'N_star.50',
 'N_star.51',
 'N_star.52',
 'N_star.53',
 'N_star.54',
 'N_star.55',
 'N_star.56',
 'N_star.57',
 'N_star.58',
 'N_star.59',
 'N_star.60',
 'N_star.61',
 'N_star.62',
 'N_star.63',

In [6]:
df = pd.read_pickle('/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/preprocessed_cleaned_all_result.pkl')

In [7]:
df

,id,time_window_start,tweets_struct,name,username,description,location,public_metrics.followers_count,profile_image_url,clean...pool,...,Belief_increase_fossil_fuel_production_Symbol,Belief_increase_fossil_fuel_production_Symbol_num,Belief_increase_fossil_fuel_production_Symbol_cat,Demo_sex_idx,Demo_age_idx,Demo_ethnicity_idx,Demo_income_idx,Demo_education_idx,Demo_past_vote_idx,state_cleaned_idx
0,1081532502966239232,2023-11-05 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1081532502966239232,2023-11-19 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1081532502966239232,2023-12-03 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1081532502966239232,2023-12-17 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1081532502966239232,2023-12-31 00:00:00+00:00,[],caleb ❍,xxaleb,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,4426,https://pbs.twimg.com/profile_images/180336437...,trad,...,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23984,142827500,2024-08-11 00:00:00+00:00,"['12 weeks to election', {'text': '@maxies_bac...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,2,2,2,1.0,2.0,1.0,4.0,2.0,3.0,15.0
23985,142827500,2024-08-25 00:00:00+00:00,"['10 weeks to election', {'text': '@foldyrhand...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,2,2,2,1.0,2.0,1.0,3.0,2.0,3.0,15.0
23987,142827500,2024-09-22 00:00:00+00:00,"['6 weeks to election', {'text': '@kevin__fm S...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,2,2,2,1.0,2.0,1.0,3.0,2.0,3.0,15.0
23988,142827500,2024-10-06 00:00:00+00:00,"['4 weeks to election', {'text': '@cherryrepre...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,2,2,2,1.0,2.0,1.0,3.0,2.0,1.0,15.0


In [8]:
print(df.columns.to_list())

['id', 'time_window_start', 'tweets_struct', 'name', 'username', 'description', 'location', 'public_metrics.followers_count', 'profile_image_url', 'clean...pool', 'is_empty_window', 'window_weeks_to_election', 'profile_image_description', 'profile_image_main_info', 'Model', 'Belief_Raw', 'Belief_Cleaned', 'Demo_Raw', 'Demo_Cleaned', 'Belief_opinion_on_the_number_of_immigrants_in_america_Category', 'Belief_opinion_on_the_number_of_immigrants_in_america_Explanation', 'Belief_opinion_on_the_number_of_immigrants_in_america_Symbol', 'Belief_opinion_on_the_number_of_immigrants_in_america_Speculation', 'Belief_opinion_on_the_statement_immigrants_increase_crime_rates_Category', 'Belief_opinion_on_the_statement_immigrants_increase_crime_rates_Explanation', 'Belief_opinion_on_the_statement_immigrants_increase_crime_rates_Symbol', 'Belief_opinion_on_the_statement_immigrants_increase_crime_rates_Speculation', 'Belief_opinion_on_the_statement_immigrants_take_jobs_away_from_people_who_were_born_in_t

In [9]:
belief_col = ['Belief_opinion_on_the_following_statement_it_should_be_acceptable_for_a_pregnant_woman_to_obtain_a_legal_abortion_if_a_woman_wants_it_for_any_reason_Symbol', 'Belief_opinion_on_funding_of_police_departments_Symbol']

In [10]:
predictor_cols = [
    "Demo_sex_Symbol",
    "Demo_age_Symbol",
    "Demo_ethnicity_Symbol",
    "Demo_income_Symbol",
    "Demo_education_Symbol",
    "Demo_past_vote_Symbol",
    "time_window_start",
    "state_cleaned"
]
df = df.dropna(subset=belief_col + predictor_cols)

In [11]:
df

,id,time_window_start,tweets_struct,name,username,description,location,public_metrics.followers_count,profile_image_url,clean...pool,...,Belief_increase_fossil_fuel_production_Symbol,Belief_increase_fossil_fuel_production_Symbol_num,Belief_increase_fossil_fuel_production_Symbol_cat,Demo_sex_idx,Demo_age_idx,Demo_ethnicity_idx,Demo_income_idx,Demo_education_idx,Demo_past_vote_idx,state_cleaned_idx
40,1542629834953719810,2024-05-19 00:00:00+00:00,"['24 weeks to election', {'text': '@The_Trump_...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,1,1,1,2.0,3.0,1.0,3.0,1.0,2.0,36.0
41,1542629834953719810,2024-06-16 00:00:00+00:00,"['20 weeks to election', {'text': '🤣 https://t...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,1,1,1,2.0,3.0,1.0,3.0,1.0,2.0,36.0
43,1542629834953719810,2024-07-14 00:00:00+00:00,"['16 weeks to election', {'text': '🤣🤣🤣 https:/...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,1,1,1,2.0,3.0,1.0,3.0,1.0,2.0,36.0
44,1542629834953719810,2024-07-28 00:00:00+00:00,"['14 weeks to election', {'text': '@victormarx...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,2,2,2,2.0,3.0,1.0,2.0,1.0,3.0,36.0
45,1542629834953719810,2024-08-11 00:00:00+00:00,"['12 weeks to election', {'text': '🤣🥰 https://...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,1,1,1,2.0,3.0,1.0,3.0,1.0,2.0,36.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23984,142827500,2024-08-11 00:00:00+00:00,"['12 weeks to election', {'text': '@maxies_bac...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,2,2,2,1.0,2.0,1.0,4.0,2.0,3.0,15.0
23985,142827500,2024-08-25 00:00:00+00:00,"['10 weeks to election', {'text': '@foldyrhand...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,2,2,2,1.0,2.0,1.0,3.0,2.0,3.0,15.0
23987,142827500,2024-09-22 00:00:00+00:00,"['6 weeks to election', {'text': '@kevin__fm S...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,2,2,2,1.0,2.0,1.0,3.0,2.0,3.0,15.0
23988,142827500,2024-10-06 00:00:00+00:00,"['4 weeks to election', {'text': '@cherryrepre...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,2,2,2,1.0,2.0,1.0,3.0,2.0,1.0,15.0


In [12]:
df['time_window_start'] = pd.to_datetime(df['time_window_start'])
time_window_sorted = df['time_window_start'].unique()
time_window_sorted = sorted(time_window_sorted)
time_window_map = {window: idx + 1 for idx, window in enumerate(time_window_sorted)}
df.loc[:, 'time_window_start_idx'] = df['time_window_start'].map(time_window_map)

/tmp/ipykernel_572405/1803766872.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['time_window_start'] = pd.to_datetime(df['time_window_start'])
/tmp/ipykernel_572405/1803766872.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:, 'time_window_start_idx'] = df['time_window_start'].map(time_window_map)


In [13]:
time_window_mapping = df[['time_window_start', 'time_window_start_idx']].drop_duplicates().sort_values('time_window_start_idx')
print(time_window_mapping)

            time_window_start  time_window_start_idx
233 2023-11-05 00:00:00+00:00                      1
260 2023-11-19 00:00:00+00:00                      2
131 2023-12-03 00:00:00+00:00                      3
236 2023-12-17 00:00:00+00:00                      4
237 2023-12-31 00:00:00+00:00                      5
186 2024-01-14 00:00:00+00:00                      6
239 2024-01-28 00:00:00+00:00                      7
188 2024-02-11 00:00:00+00:00                      8
59  2024-02-25 00:00:00+00:00                      9
242 2024-03-10 00:00:00+00:00                     10
191 2024-03-24 00:00:00+00:00                     11
192 2024-04-07 00:00:00+00:00                     12
193 2024-04-21 00:00:00+00:00                     13
142 2024-05-05 00:00:00+00:00                     14
40  2024-05-19 00:00:00+00:00                     15
196 2024-06-02 00:00:00+00:00                     16
41  2024-06-16 00:00:00+00:00                     17
198 2024-06-30 00:00:00+00:00                 

In [14]:
df

,id,time_window_start,tweets_struct,name,username,description,location,public_metrics.followers_count,profile_image_url,clean...pool,...,Belief_increase_fossil_fuel_production_Symbol_num,Belief_increase_fossil_fuel_production_Symbol_cat,Demo_sex_idx,Demo_age_idx,Demo_ethnicity_idx,Demo_income_idx,Demo_education_idx,Demo_past_vote_idx,state_cleaned_idx,time_window_start_idx
40,1542629834953719810,2024-05-19 00:00:00+00:00,"['24 weeks to election', {'text': '@The_Trump_...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,1,1,2.0,3.0,1.0,3.0,1.0,2.0,36.0,15
41,1542629834953719810,2024-06-16 00:00:00+00:00,"['20 weeks to election', {'text': '🤣 https://t...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,1,1,2.0,3.0,1.0,3.0,1.0,2.0,36.0,17
43,1542629834953719810,2024-07-14 00:00:00+00:00,"['16 weeks to election', {'text': '🤣🤣🤣 https:/...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,1,1,2.0,3.0,1.0,3.0,1.0,2.0,36.0,19
44,1542629834953719810,2024-07-28 00:00:00+00:00,"['14 weeks to election', {'text': '@victormarx...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,2,2,2.0,3.0,1.0,2.0,1.0,3.0,36.0,20
45,1542629834953719810,2024-08-11 00:00:00+00:00,"['12 weeks to election', {'text': '🤣🥰 https://...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,1,1,2.0,3.0,1.0,3.0,1.0,2.0,36.0,21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23984,142827500,2024-08-11 00:00:00+00:00,"['12 weeks to election', {'text': '@maxies_bac...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,2,2,1.0,2.0,1.0,4.0,2.0,3.0,15.0,21
23985,142827500,2024-08-25 00:00:00+00:00,"['10 weeks to election', {'text': '@foldyrhand...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,2,2,1.0,2.0,1.0,3.0,2.0,3.0,15.0,22
23987,142827500,2024-09-22 00:00:00+00:00,"['6 weeks to election', {'text': '@kevin__fm S...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,2,2,1.0,2.0,1.0,3.0,2.0,3.0,15.0,24
23988,142827500,2024-10-06 00:00:00+00:00,"['4 weeks to election', {'text': '@cherryrepre...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,2,2,1.0,2.0,1.0,3.0,2.0,1.0,15.0,25


In [15]:
# prepare spatial structure for ICAR
def edges_from_adjacency(W):
    all_edges = []
    n = W.shape[0]
    
    for i in range(n):
        neighbors = np.where(W[i, :] != 0)[0]
        for j in neighbors:
            if i < j:  
                all_edges.append({
                    'node1': i + 1,  
                    'node2': j + 1,
                    'weight': W[i, j]
                })
    
    return pd.DataFrame(all_edges)


def scale_c(C):
    N = C.shape[0]
    
    if hasattr(C, 'toarray'):
        C = C.toarray()
    C = np.asarray(C)
    
    row_sums = C.sum(axis=1).ravel()
    Q = diags(row_sums) - C
    
    Q_dense = Q.toarray() if hasattr(Q, 'toarray') else np.asarray(Q)
    max_diag = np.max(np.diag(Q_dense))
    jitter = max_diag * np.sqrt(np.finfo(float).eps)
    Q_pert = Q_dense + np.diag(np.ones(N) * jitter)
    
    Q_inv = np.linalg.pinv(Q_pert)
    
    variances = np.diag(Q_inv)
    scaling_factor = np.exp(np.mean(np.log(variances)))
    
    return scaling_factor


def prep_icar_data_from_weights(w, N_states):
    C = np.zeros((N_states, N_states))
    for i in range(N_states):
        for j in w.neighbors[i]:
            C[i, j] = 1

    E = edges_from_adjacency(C)
    
    G = nx.Graph()
    G.add_nodes_from(range(N_states))
    for _, row in E.iterrows():
        G.add_edge(int(row['node1']) - 1, int(row['node2']) - 1)
    
    components = list(nx.connected_components(G))
    k = len(components)
    
    comp_id = np.zeros(N_states, dtype=int)
    for idx, component in enumerate(components):
        for node in component:
            comp_id[int(node)] = idx
    
    group_size = np.array([len(comp) for comp in components])
    
    group_idx = comp_id + 1

    
    return {
        'k': int(k),
        'group_size': group_size.tolist(),
        'N_edges': len(E),
        'node1': E['node1'].values.tolist(),
        'node2': E['node2'].values.tolist(),
        'group_idx': group_idx.tolist()
    }


def moran_coefficient(x, W, warn=True):
    row_sums = W.sum(axis=1)
    zero_idx = np.where(row_sums == 0)[0]
    
    if len(zero_idx) > 0:
        if warn:
            print(f"{len(zero_idx)} observations with no neighbors. Dropping from calculation.")
        keep_idx = row_sums > 0
        x = x[keep_idx]
        W = W[np.ix_(keep_idx, keep_idx)]
    
    n = len(x)
    xbar = np.mean(x)
    z = x - xbar
    z_tilde = W @ z
    A = np.sum(W)
    mc = (n / A) * (z @ z_tilde) / (z @ z)
    
    return np.round(mc, 3)

# Build spatial structure
shp = gpd.read_file("/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/tl_2025_us_state/tl_2025_us_state.shp")
shp["STATE"] = shp["STUSPS"].str.upper()

shp = shp[~shp["STATE"].isin(["PR", "GU", "VI", "MP", "AS"])]

shp = shp.sort_values("STATE").reset_index(drop=True)

state_order = shp["STATE"].tolist()
N_states = len(state_order)

print(f"Shapefile states: {N_states}")

w = weights.Queen.from_dataframe(shp)

icar_data = prep_icar_data_from_weights(w, N_states)

print(f"\nICAR structure:")
print(f"  Connected components: {icar_data['k']}")
print(f"  Component sizes: {icar_data['group_size']}")
print(f"  Number of edges: {icar_data['N_edges']}")



Shapefile states: 51


/tmp/ipykernel_572405/4196857512.py:111: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = weights.Queen.from_dataframe(shp)



ICAR structure:
  Connected components: 3
  Component sizes: [1, 49, 1]
  Number of edges: 112


/home/yuyue/stan_env/lib/python3.12/site-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 3 disconnected components.
 There are 2 islands with ids: 0, 11.
  W.__init__(self, neighbors, ids=ids, **kw)


In [16]:
shp.head()

,REGION,DIVISION,STATEFP,STATENS,GEOID,GEOIDFQ,STUSPS,NAME,LSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry,STATE
0,4,9,02,01785533,02,0400000US02,AK,Alaska,00,G4000,A,1479893380150,244326118163,+63.3473560,-152.8397334,"MULTIPOLYGON (((-167.55823 60.22436, -167.5567...",AK
1,3,6,01,01779775,01,0400000US01,AL,Alabama,00,G4000,A,131186429591,4580729056,+32.7395785,-086.8434469,"POLYGON ((-85.12733 31.76256, -85.12753 31.762...",AL
2,3,7,05,00068085,05,0400000US05,AR,Arkansas,00,G4000,A,134658642291,3122591253,+34.8955256,-092.4446262,"POLYGON ((-90.95577 34.11871, -90.95451 34.117...",AR
3,4,8,04,01779777,04,0400000US04,AZ,Arizona,00,G4000,A,294366238828,853868947,+34.2039362,-111.6063449,"POLYGON ((-110.75069 37.00301, -110.74193 37.0...",AZ
4,4,9,06,01779778,06,0400000US06,CA,California,00,G4000,A,403676564152,20288505023,+37.1551773,-119.5434183,"MULTIPOLYGON (((-119.99987 41.18397, -119.9998...",CA


In [17]:
strat_frame[f'{belief_col}_LP']

0         0.002266
1         0.002266
2         0.002266
3         0.002266
4         0.002266
            ...   
190839    2.260323
190840    2.260323
190841    2.260323
190842    2.260323
190843    2.260323
Name: Belief_opinion_on_the_number_of_immigrants_in_america_Symbol_LP, Length: 190844, dtype: float64

here make the belief_lp_col include both f'{belief_col}_LP' and f'{belief_col}_LP_2' if any of them exist, if both exist then include both

In [18]:
# state level fixed effects
base_lp_col = 'pct.2020.votes.R-D_zscore'
belief_lp_candidates = [f'{belief_col}_LP', f'{belief_col}_LP_2']

LP_cols = [base_lp_col] + [c for c in belief_lp_candidates if c in strat_frame.columns]

LP_state = (
    strat_frame
    .assign(STATE=strat_frame["state_abbreviation"].str.upper())
    .groupby("STATE")[LP_cols]
    .mean()
)
LP_state = LP_state.reindex(state_order)

LP_mat = LP_state.values
P = LP_mat.shape[1]

print("LP_cols:", LP_cols)
print("LP_mat shape:", LP_mat.shape)


LP_cols: ['pct.2020.votes.R-D_zscore', 'Belief_opinion_on_the_number_of_immigrants_in_america_Symbol_LP']
LP_mat shape: (51, 2)


In [19]:
print(LP_mat)

[[ 3.98625124e-01  2.26595691e-03]
 [ 1.04702954e+00  1.05000459e+00]
 [ 1.13793037e+00  4.35812979e-01]
 [-3.79779657e-02 -2.35404134e-02]
 [-1.25283723e+00 -4.94867962e-01]
 [-5.89451771e-01 -4.82685969e-01]
 [-8.70138114e-01 -1.06633586e+00]
 [-3.67750855e+00 -3.40990968e+00]
 [-8.23594001e-01 -1.38730783e+00]
 [ 1.16416745e-01  1.22124928e-01]
 [-3.48990760e-02  3.70928391e-01]
 [-1.25416105e+00 -4.14606180e-01]
 [ 3.18283229e-01 -3.65146774e-01]
 [ 1.27057540e+00 -3.20313673e-01]
 [-7.40232980e-01 -1.91281821e-01]
 [ 6.51170633e-01  8.88022887e-02]
 [ 5.91951976e-01  6.59155384e-01]
 [ 1.06696123e+00  9.21911155e-01]
 [ 7.58530759e-01  1.46571582e+00]
 [-1.42342154e+00 -1.45556760e+00]
 [-1.42323407e+00 -1.54928269e+00]
 [-4.02826779e-01  1.31297809e-01]
 [-1.42175029e-01 -2.96437700e-01]
 [-3.24338755e-01 -1.15802404e-01]
 [ 6.23020341e-01  1.03325869e-01]
 [ 6.71627070e-01  7.24844327e-01]
 [ 6.64206504e-01  1.71363578e+00]
 [ 3.17794270e-02  2.80180715e-01]
 [ 1.37945875e+00  1

In [20]:
LP_df = pd.DataFrame(
    LP_mat,
    index=[s.upper() for s in state_order],
    columns=LP_cols
)

LP_df.head()

,pct.2020.votes.R-D_zscore,Belief_opinion_on_the_number_of_immigrants_in_america_Symbol_LP
AK,0.398625,0.002266
AL,1.047030,1.050005
AR,1.137930,0.435813
AZ,-0.037978,-0.023540
CA,-1.252837,-0.494868


In [21]:
print(df["Demo_age_idx"].unique())

[3. 4. 5. 2. 6. 1.]


In [22]:
# Stan code
stan_code = '''
// function for ICAR prior
functions {
  real icar_normal_lpdf(vector phi, int N_edges,
                        array[] int node1, array[] int node2) {
    return -0.5 * dot_self(phi[node1] - phi[node2]);
  }
}

data {
  int<lower=1> N;               // number of survey respondents
  int<lower=2> K;               // number of belief categories
  array[N] int<lower=1, upper=K> y; //outcome variable(order 1..K)

  // levels for each variable
  int<lower=1> J_sex;           
  int<lower=1> J_age;
  int<lower=1> J_eth;
  int<lower=1> J_income;
  int<lower=1> J_education;
  int<lower=1> J_vote;
  int<lower=1> J_time;
  int<lower=1> N_states;

  // map each respondent to their demographic levels
  array[N] int<lower=1, upper=J_sex> sex_idx; 
  array[N] int<lower=1, upper=J_age> age_idx;
  array[N] int<lower=1, upper=J_eth> eth_idx;
  array[N] int<lower=1, upper=J_income> income_idx;
  array[N] int<lower=1, upper=J_education> education_idx;
  array[N] int<lower=1, upper=J_vote> vote_idx;
  array[N] int<lower=1, upper=J_time> time_idx;
  array[N] int<lower=1, upper=N_states> state_idx;

  // ICAR structure
  int<lower=1> N_edges;
  array[N_edges] int node1;
  array[N_edges] int node2;

  int<lower=1> k;                           // number of connected components
  array[k] int group_size;                  // size of each connected component
  array[N_states] int group_idx;           // component assignment for each state

  // state level fixed effects
  int<lower=1> P;
  matrix[N_states, P] LP;

  // Poststratification prediction rows

  int<lower=1> N_pred;
  array[N_pred] int<lower=1, upper=J_sex> pred_sex_idx;
  array[N_pred] int<lower=1, upper=J_age> pred_age_idx;
  array[N_pred] int<lower=1, upper=J_eth> pred_eth_idx;
  array[N_pred] int<lower=1, upper=J_income> pred_income_idx;
  array[N_pred] int<lower=1, upper=J_education> pred_education_idx;
  array[N_pred] int<lower=1, upper=J_vote> pred_vote_idx;
  array[N_pred] int<lower=1, upper=J_time> pred_time_idx;
  array[N_pred] int<lower=1, upper=N_states> pred_state_idx;
}

parameters {
  ordered[K-1] kappa;

  sum_to_zero_vector[J_sex] eta_sex; 
  sum_to_zero_vector[J_age] eta_age;             
  sum_to_zero_vector[J_eth] eta_eth;
  sum_to_zero_vector[J_income] eta_income; 
  sum_to_zero_vector[J_education] eta_education;     
  sum_to_zero_vector[J_vote] eta_vote;
  sum_to_zero_vector[J_time] eta_time;

  // Two-way interactions with time
  sum_to_zero_matrix[J_time, J_sex] eta_time_sex;
  sum_to_zero_matrix[J_time, J_age] eta_time_age;
  sum_to_zero_matrix[J_time, J_eth] eta_time_eth;
  sum_to_zero_matrix[J_time, J_income] eta_time_income;
  sum_to_zero_matrix[J_time, J_education] eta_time_education;
  sum_to_zero_matrix[J_time, J_vote] eta_time_vote;

  vector[N_states] eta_state_raw;
  vector[P] beta_LP;

  real<lower=0> sigma_sex;
  real<lower=0> sigma_age;
  real<lower=0> sigma_eth;
  real<lower=0> sigma_income;
  real<lower=0> sigma_education;
  real<lower=0> sigma_vote;
  real<lower=0> sigma_time;
  real<lower=0> sigma_state;

  // Interaction scales
  real<lower=0> sigma_time_sex;
  real<lower=0> sigma_time_age;
  real<lower=0> sigma_time_eth;
  real<lower=0> sigma_time_income;
  real<lower=0> sigma_time_education;
  real<lower=0> sigma_time_vote;
}

transformed parameters {
  vector[J_sex] eta_sex_star = eta_sex * sigma_sex;
  vector[J_age] eta_age_star = eta_age * sigma_age;
  vector[J_eth] eta_eth_star = eta_eth * sigma_eth;
  vector[J_income] eta_income_star = eta_income * sigma_income;
  vector[J_education] eta_education_star = eta_education * sigma_education;
  vector[J_vote] eta_vote_star = eta_vote * sigma_vote;
  vector[J_time] eta_time_star = eta_time * sigma_time;

  matrix[J_time, J_sex] eta_time_sex_star = eta_time_sex * sigma_time_sex;
  matrix[J_time, J_age] eta_time_age_star = eta_time_age * sigma_time_age;
  matrix[J_time, J_eth] eta_time_eth_star = eta_time_eth * sigma_time_eth;
  matrix[J_time, J_income] eta_time_income_star = eta_time_income * sigma_time_income;
  matrix[J_time, J_education] eta_time_education_star = eta_time_education * sigma_time_education;
  matrix[J_time, J_vote] eta_time_vote_star = eta_time_vote * sigma_time_vote;
  vector[N_states] mu_state_LP = LP * beta_LP;
  vector[N_states] eta_state = sigma_state * eta_state_raw;
}

model {
  // Priors
  kappa ~ normal(0, 3);
  beta_LP ~ normal(0, 1);

   // prior on the constrained vector 
  eta_sex ~ normal(0, 1);
  eta_eth ~ normal(0, 1);
  eta_vote ~ normal(0, 1);
  eta_education ~ normal(0, 1);

  // prior on the interaction terms
  to_vector(eta_time_sex) ~ normal(0, 1);
  to_vector(eta_time_age) ~ normal(0, 1);
  to_vector(eta_time_eth) ~ normal(0, 1);
  to_vector(eta_time_income) ~ normal(0, 1);
  to_vector(eta_time_education) ~ normal(0, 1);
  to_vector(eta_time_vote) ~ normal(0, 1);

  sigma_sex ~ normal(0, 1);
  sigma_age ~ normal(0, 1);
  sigma_eth ~ normal(0, 1);
  sigma_income ~ normal(0, 1);
  sigma_education ~ normal(0, 1);
  sigma_vote ~ normal(0, 1);
  sigma_time ~ normal(0, 1);
  sigma_state ~ normal(0, 1);

  sigma_time_sex ~ normal(0, 1);
  sigma_time_age ~ normal(0, 1);
  sigma_time_eth ~ normal(0, 1);
  sigma_time_income ~ normal(0, 1);
  sigma_time_education ~ normal(0, 1);
  sigma_time_vote ~ normal(0, 1);

  for (i in 2:J_age) {
    eta_age[i] ~ normal(eta_age[i-1], 1);
  }
  for (i in 2:J_income) {
    eta_income[i] ~ normal(eta_income[i-1], 1);
  }
  for (i in 2:J_time) {
    eta_time[i] ~ normal(eta_time[i-1], 1);
  }

  // ICAR prior on raw spatial effects
  target += icar_normal_lpdf(eta_state_raw | N_edges, node1, node2);
  
  // Sum-to-zero constraint per connected component (island can have free prior)
  for (j in 1:k) {
  if (group_size[j] > 1) {
    // Soft sum-to-zero constraint for proper ICAR within this component
    real component_sum = 0;
    for (i in 1:N_states) {
      if (group_idx[i] == j) {
        component_sum += eta_state_raw[i];
      }
    }
    target += normal_lpdf(component_sum | 0, 0.001 * group_size[j]);
  } else {
    // Island component: give the singleton node a proper iid prior
    for (i in 1:N_states) {
      if (group_idx[i] == j) {
        eta_state_raw[i] ~ normal(0, 1);
      }
    }
  }
}

  // Ordered logistic likelihood
  for (i in 1:N) {
    real mu = eta_sex_star[sex_idx[i]]
              + eta_age_star[age_idx[i]]
              + eta_eth_star[eth_idx[i]]
              + eta_income_star[income_idx[i]]
              + eta_vote_star[vote_idx[i]]
              + eta_education_star[education_idx[i]]
              + eta_time_star[time_idx[i]]
              + eta_time_sex_star[time_idx[i], sex_idx[i]]
              + eta_time_age_star[time_idx[i], age_idx[i]]
              + eta_time_eth_star[time_idx[i], eth_idx[i]]
              + eta_time_income_star[time_idx[i], income_idx[i]]
              + eta_time_education_star[time_idx[i], education_idx[i]]
              + eta_time_vote_star[time_idx[i], vote_idx[i]]
              + eta_state[state_idx[i]]
              + mu_state_LP[state_idx[i]];
    y[i] ~ ordered_logistic(mu, kappa);
  }
}

generated quantities {

  vector[N_pred] mu_pred;
  vector[N_pred] expected_y_pred;
  array[N_pred] simplex[K] prob_pred;

  for (i in 1:N_pred) {
    mu_pred[i] = eta_sex_star[pred_sex_idx[i]]
                 + eta_age_star[pred_age_idx[i]]
                 + eta_eth_star[pred_eth_idx[i]]
                 + eta_income_star[pred_income_idx[i]]
                 + eta_vote_star[pred_vote_idx[i]]
                 + eta_education_star[pred_education_idx[i]]
                 + eta_time_star[pred_time_idx[i]]
                 + eta_time_sex_star[pred_time_idx[i], pred_sex_idx[i]]
                 + eta_time_age_star[pred_time_idx[i], pred_age_idx[i]]
                 + eta_time_eth_star[pred_time_idx[i], pred_eth_idx[i]]
                 + eta_time_income_star[pred_time_idx[i], pred_income_idx[i]]
                 + eta_time_education_star[pred_time_idx[i], pred_education_idx[i]]
                 + eta_time_vote_star[pred_time_idx[i], pred_vote_idx[i]]
                 + eta_state[pred_state_idx[i]]
                 + mu_state_LP[pred_state_idx[i]];

    prob_pred[i, 1] = inv_logit(kappa[1] - mu_pred[i]);
    for (k_cat in 2:(K - 1)) {
      prob_pred[i, k_cat] = inv_logit(kappa[k_cat] - mu_pred[i]) - inv_logit(kappa[k_cat - 1] - mu_pred[i]);
    }
    prob_pred[i, K] = 1 - inv_logit(kappa[K - 1] - mu_pred[i]);

    expected_y_pred[i] = 0;
    for (k_cat in 1:K) {
      expected_y_pred[i] += k_cat * prob_pred[i, k_cat];
    }
  }
}
'''

# Compile and Sample

with open('mrp.stan', 'w') as f:
    f.write(stan_code)

model = CmdStanModel(stan_file='mrp.stan')


20:42:34 - cmdstanpy - INFO - compiling stan file /tmp/tmp3iswfpbn/tmpx4i0hkt5.stan to exe file /mnt/c/Users/yuyue/AppData/Local/Programs/Microsoft VS Code/mrp
20:42:53 - cmdstanpy - INFO - compiled model executable: /mnt/c/Users/yuyue/AppData/Local/Programs/Microsoft VS Code/mrp


In [23]:
# idx = (time_idx - 1) * J_predictor + predictor_idx
J_sex = int(strat_frame['sex_idx'].nunique())
J_age = int(strat_frame['age_idx'].nunique())
J_eth = int(strat_frame['eth_idx'].nunique())
J_income = int(strat_frame['income_idx'].nunique())
J_vote = int(strat_frame['vote_idx'].nunique())
J_education = int(strat_frame['education_idx'].nunique())
J_time = int(df['time_window_start'].nunique())

time_idx_arr = df['time_window_start_idx'].astype(int)
sex_idx_arr = df['Demo_sex_idx'].astype(int)
age_idx_arr = df['Demo_age_idx'].astype(int)
eth_idx_arr = df['Demo_ethnicity_idx'].astype(int)
income_idx_arr = df['Demo_income_idx'].astype(int)
vote_idx_arr = df['Demo_past_vote_idx'].astype(int)
education_idx_arr = df['Demo_education_idx'].astype(int)
state_idx_arr = df['state_cleaned_idx'].astype(int)

# Prediction rows: one row per poststrat cell for this window
N_pred = len(strat_frame)
pred_sex_idx_arr = strat_frame['sex_idx'].astype(int)
pred_age_idx_arr = strat_frame['age_idx'].astype(int)
pred_eth_idx_arr = strat_frame['eth_idx'].astype(int)
pred_income_idx_arr = strat_frame['income_idx'].astype(int)
pred_vote_idx_arr = strat_frame['vote_idx'].astype(int)
pred_education_idx_arr = strat_frame['education_idx'].astype(int)
pred_state_idx_arr = strat_frame['state_idx'].astype(int)


In [24]:

# Build one base data dict for full-data training
stan_data_base = {
    "N": len(df),
    "K": df[belief_col + "_cat"].cat.categories.size,
    "y": df[belief_col + "_num"].astype(int).tolist(),

    # Predictor dimensions
    "J_sex": J_sex,
    "J_age": J_age,
    "J_eth": J_eth,
    "J_income": J_income,
    "J_vote": J_vote,
    "J_education": J_education,
    "J_time": J_time,
    "N_states": N_states,

    # Training indices
    "sex_idx": sex_idx_arr.tolist(),
    "age_idx": age_idx_arr.tolist(),
    "eth_idx": eth_idx_arr.tolist(),
    "income_idx": income_idx_arr.tolist(),
    "vote_idx": vote_idx_arr.tolist(),
    "education_idx": education_idx_arr.tolist(),
    "state_idx": state_idx_arr.tolist(),
    "time_idx": time_idx_arr.tolist(),


    # prediction rows(pred_time_idx is initially set to 1 for all and will be updated in loop)
    "N_pred": N_pred,
    "pred_sex_idx": pred_sex_idx_arr.tolist(),
    "pred_age_idx": pred_age_idx_arr.tolist(),
    "pred_eth_idx": pred_eth_idx_arr.tolist(),
    "pred_income_idx": pred_income_idx_arr.tolist(),
    "pred_education_idx": pred_education_idx_arr.tolist(),
    "pred_vote_idx": pred_vote_idx_arr.tolist(),
    "pred_time_idx": np.ones(N_pred, dtype=int).tolist(),
    "pred_state_idx": pred_state_idx_arr.tolist(),

    # ICAR spatial structure
    **icar_data,

    # State-level fixed effect
    "P": LP_mat.shape[1],
    "LP": LP_mat.tolist(),
}


In [25]:

base_dir = f"/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/{belief_col}"
os.makedirs(base_dir, exist_ok=True)

global_model_dir = f"{base_dir}/all_time_train_stan_output"
os.makedirs(global_model_dir, exist_ok=True)

# ----- Fit on full training data -----
print(f"Fitting full model once: N={len(df)}, J_time={J_time}")
fit = model.sample(
    data=stan_data_base,
    chains=8,
    parallel_chains=8,
    iter_warmup=2000,
    iter_sampling=1000,
    adapt_delta=0.995,
    max_treedepth=13,
    thin=32,
    sig_figs=18,
    show_console=True,
    output_dir=global_model_dir
)


print(fit.summary())
summary_df = fit.summary()
summary_df.to_csv(f"{global_model_dir}/mrp_summary.csv", index=True)
with open(f"{global_model_dir}/fit_full_train.pkl", "wb") as f:
    pickle.dump(fit, f)
print(f"Saved full-training fit to: {global_model_dir}")


Fitting full model once: N=7356, J_time=26


21:42:51 - cmdstanpy - INFO - Chain [1] start processing
21:42:51 - cmdstanpy - INFO - Chain [2] start processing
21:42:51 - cmdstanpy - INFO - Chain [3] start processing
21:42:51 - cmdstanpy - INFO - Chain [4] start processing
21:42:51 - cmdstanpy - INFO - Chain [5] start processing
21:42:51 - cmdstanpy - INFO - Chain [6] start processing
21:42:51 - cmdstanpy - INFO - Chain [7] start processing
21:42:51 - cmdstanpy - INFO - Chain [8] start processing


Chain [2] method = sample (Default)Chain [3] method = sample (Default)
Chain [8] method = sample (Default)
Chain [8] sample
Chain [1] method = sample (Default)
Chain [1] sample
Chain [1] num_samples = 1000 (Default)
Chain [1] num_warmup = 2000
Chain [1] save_warmup = false (Default)
Chain [3] sample
Chain [3] num_samples = 1000 (Default)
Chain [8] num_samples = 1000 (Default)
Chain [8] num_warmup = 2000
Chain [8] save_warmup = false (Default)
Chain [8] thin = 32
Chain [8] adapt
Chain [8] engaged = true (Default)
Chain [8] gamma = 0.05 (Default)
Chain [8] delta = 0.995
Chain [8] kappa = 0.75 (Default)
Chain [8] t0 = 10 (Default)
Chain [8] init_buffer = 75 (Default)
Chain [8] term_buffer = 50 (Default)
Chain [8] window = 25 (Default)
Chain [8] save_metric = false (Default)
Chain [8] algorithm = hmc (Default)
Chain [8] hmc
Chain [8] engine = nuts (Default)
Chain [8] nuts
Chain [8] max_depth = 13
Chain [8] metric = diag_e (Default)
Chain [8] metric_file =  (Default)
Chain [8] stepsize = 1 

23:22:00 - cmdstanpy - INFO - Chain [7] done processing


Chain [7] 
Chain [7] Elapsed Time: 4387.29 seconds (Warm-up)
Chain [7] 1738.69 seconds (Sampling)
Chain [7] 6125.98 seconds (Total)
Chain [7] 
Chain [7] 
Chain [7] 
Chain [7] 
Chain [5] Iteration: 2900 / 3000 [ 96%]  (Sampling)
Chain [2] Iteration: 2400 / 3000 [ 80%]  (Sampling)
Chain [3] Iteration: 2400 / 3000 [ 80%]  (Sampling)
Chain [6] Iteration: 2500 / 3000 [ 83%]  (Sampling)
Chain [4] Iteration: 2400 / 3000 [ 80%]  (Sampling)
Chain [8] Iteration: 2800 / 3000 [ 93%]  (Sampling)
Chain [1] Iteration: 2300 / 3000 [ 76%]  (Sampling)
Chain [5] Iteration: 3000 / 3000 [100%]  (Sampling)


23:25:13 - cmdstanpy - INFO - Chain [5] done processing


... (9 outputs left out)

23:51:47 - cmdstanpy - INFO - Chain [4] done processing


Chain [4] 
Chain [4] Elapsed Time: 4879.12 seconds (Warm-up)
Chain [4] 3088.27 seconds (Sampling)
Chain [4] 7967.39 seconds (Total)
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [4] 
Chain [1] Iteration: 2900 / 3000 [ 96%]  (Sampling)
Chain [1] Iteration: 3000 / 3000 [100%]  (Sampling)


23:55:36 - cmdstanpy - INFO - Chain [1] done processing
23:55:36 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: ordered_logistic: Location parameter is -inf, but must be finite! (in 'mrp.stan', line 206, column 4 to column 39)
	Exception: ordered_logistic: Location parameter is -inf, but must be finite! (in 'mrp.stan', line 206, column 4 to column 39)
	Exception: ordered_logistic: Cut-points is not a valid ordered vector. The element at 2 is -395.906, but should be greater than the previous element, -395.906 (in 'mrp.stan', line 206, column 4 to column 39)
	Exception: ordered_logistic: Cut-points is not a valid ordered vector. The element at 2 is -100.546, but should be greater than the previous element, -100.546 (in 'mrp.stan', line 206, column 4 to column 39)
	Exception: ordered_logistic: Cut-points is not a valid ordered vector. The element at 2 is -26.3464, but should be greater than the previous element, -26.3464 (in 'mrp.stan', line 206, column 4 to column 39

Chain [1] 
Chain [1] Elapsed Time: 5263.58 seconds (Warm-up)
Chain [1] 2939.82 seconds (Sampling)
Chain [1] 8203.41 seconds (Total)
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 

# Get Predictions

In [45]:
strat_frame

,Unnamed: 0.1,Unnamed: 0,sex_idx,age_idx,eth_idx,income_idx,state_idx,vote_idx,education_idx,N_star.1,...,Belief_level_of_concern_about_the_possibility_of_ukraine_being_defeated_and_taken_over_by_russia_Symbol_LP,Belief_opinion_on_whether_the_us_has_a_responsibility_to_help_ukraine_defend_itself_from_russia_s_invasion_Symbol_LP,Belief_opinion_on_the_following_statement_it_should_be_acceptable_for_a_pregnant_woman_to_obtain_a_legal_abortion_if_a_woman_wants_it_for_any_reason_Symbol_LP,Belief_opinion_on_the_statement_a_person_should_require_a_police_permit_before_he_or_she_could_buy_a_gun_Symbol_LP,Belief_opinion_on_the_statement_the_use_of_puberty_blockers_and_other_genderaffirming_procedures_for_minors_should_be_allowed_Symbol_LP,Belief_increase_fossil_fuel_production_Symbol_LP,Belief_opinion_on_funding_of_police_departments_Symbol_LP,Belief_opinion_on_whether_white_people_benefit_from_societal_advantages_not_available_to_black_people_Symbol_LP,Belief_opinion_on_the_statement_the_us_should_provide_military_aid_to_israel_to_help_in_its_war_against_hamas_Symbol_LP_2,Belief_opinion_on_the_statement_the_us_should_provide_military_aid_to_israel_to_help_in_its_war_against_hamas_Symbol_LP
0,0,0,2,1,4,1,1,1,1,11.20473,...,0.094125,0.399268,-1.207061,-2.671429,-1.563661,0.595799,-0.422827,-0.481602,-0.487338,0.133341
1,1,1,2,1,4,1,1,2,1,0.08251,...,0.094125,0.399268,-1.207061,-2.671429,-1.563661,0.595799,-0.422827,-0.481602,-0.487338,0.133341
2,2,2,2,1,4,1,1,3,1,1.58995,...,0.094125,0.399268,-1.207061,-2.671429,-1.563661,0.595799,-0.422827,-0.481602,-0.487338,0.133341
3,3,3,2,1,4,1,1,4,1,0.01103,...,0.094125,0.399268,-1.207061,-2.671429,-1.563661,0.595799,-0.422827,-0.481602,-0.487338,0.133341
4,4,4,2,1,4,1,1,6,1,0.09892,...,0.094125,0.399268,-1.207061,-2.671429,-1.563661,0.595799,-0.422827,-0.481602,-0.487338,0.133341
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190839,190839,190839,1,6,1,5,51,3,2,183.35671,...,-1.985449,-1.998531,-1.437020,-4.230682,-1.021586,0.451635,0.910535,-1.525207,-1.526627,-2.663479
190840,190840,190840,1,6,1,5,51,4,1,0.27655,...,-1.985449,-1.998531,-1.437020,-4.230682,-1.021586,0.451635,0.910535,-1.525207,-1.526627,-2.663479
190841,190841,190841,1,6,1,5,51,4,2,0.94979,...,-1.985449,-1.998531,-1.437020,-4.230682,-1.021586,0.451635,0.910535,-1.525207,-1.526627,-2.663479
190842,190842,190842,1,6,1,5,51,6,1,0.47236,...,-1.985449,-1.998531,-1.437020,-4.230682,-1.021586,0.451635,0.910535,-1.525207,-1.526627,-2.663479


In [26]:
base_dir = f"/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/{belief_col}"
global_model_dir = f"{base_dir}/all_time_train_stan_output"
print(global_model_dir)

/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/Belief_opinion_on_the_number_of_immigrants_in_america_Symbol/all_time_train_stan_output


In [27]:
fit = pickle.load(open(f"{global_model_dir}/fit_full_train.pkl", "rb"))

In [28]:
def generate_cell_level_predictions_from_gq(
    gq,
    strat_frame,
    belief_col,
    n_keep_draws=250
):
    n_cells = len(strat_frame)
    
    # Extract draws: shape (total_draws, n_cells)
    expected_y_draws = gq.stan_variable('expected_y_pred')  # (draws, N_pred)
    prob_draws = gq.stan_variable('prob_pred')               # (draws, N_pred, K)
    
    n_draws_total = expected_y_draws.shape[0]
    K = prob_draws.shape[2]
    
    # Subsample draws
    draw_indices = np.sort(np.random.choice(n_draws_total, size=min(n_keep_draws, n_draws_total), replace=False))
    n_draws = len(draw_indices)
    
    expected_y_draws = expected_y_draws[draw_indices, :]   # (n_draws, n_cells)
    prob_draws = prob_draws[draw_indices, :, :]             # (n_draws, n_cells, K)
    
    print(f"Subsampling draws: using {n_draws} out of {n_draws_total} posterior draws")
    
    cell_id = np.arange(n_cells, dtype=int)
    
    # Transpose for percentile calcs: (n_cells, n_draws)
    ev = expected_y_draws.T
    pr = prob_draws.transpose(1, 0, 2)  # (n_cells, n_draws, K)
    
    # --- Summary stats for expected value ---
    prediction_summary_df = pd.DataFrame({
        'cell_id': cell_id,
        f'{belief_col}_pred_mean':     np.mean(ev, axis=1),
        f'{belief_col}_pred_median':   np.median(ev, axis=1),
        f'{belief_col}_pred_sd':       np.std(ev, axis=1),
        f'{belief_col}_pred_lower_95': np.percentile(ev, 2.5, axis=1),
        f'{belief_col}_pred_upper_95': np.percentile(ev, 97.5, axis=1),
        f'{belief_col}_pred_lower_90': np.percentile(ev, 5, axis=1),
        f'{belief_col}_pred_upper_90': np.percentile(ev, 95, axis=1),
        f'{belief_col}_pred_lower_80': np.percentile(ev, 10, axis=1),
        f'{belief_col}_pred_upper_80': np.percentile(ev, 90, axis=1),
        f'{belief_col}_pred_lower_68': np.percentile(ev, 16, axis=1),
        f'{belief_col}_pred_upper_68': np.percentile(ev, 84, axis=1),
    })
    
    # --- Per-category summary stats ---
    for k in range(K):
        pk = pr[:, :, k]  # (n_cells, n_draws)
        cat = k + 1
        prediction_summary_df[f'{belief_col}_prob_cat_mean_{cat}']     = np.mean(pk, axis=1)
        prediction_summary_df[f'{belief_col}_prob_cat_median_{cat}']   = np.median(pk, axis=1)
        prediction_summary_df[f'{belief_col}_prob_cat_sd_{cat}']       = np.std(pk, axis=1)
        prediction_summary_df[f'{belief_col}_prob_cat_lower_95_{cat}'] = np.percentile(pk, 2.5, axis=1)
        prediction_summary_df[f'{belief_col}_prob_cat_upper_95_{cat}'] = np.percentile(pk, 97.5, axis=1)
        prediction_summary_df[f'{belief_col}_prob_cat_lower_90_{cat}'] = np.percentile(pk, 5, axis=1)
        prediction_summary_df[f'{belief_col}_prob_cat_upper_90_{cat}'] = np.percentile(pk, 95, axis=1)
        prediction_summary_df[f'{belief_col}_prob_cat_lower_80_{cat}'] = np.percentile(pk, 10, axis=1)
        prediction_summary_df[f'{belief_col}_prob_cat_upper_80_{cat}'] = np.percentile(pk, 90, axis=1)
        prediction_summary_df[f'{belief_col}_prob_cat_lower_68_{cat}'] = np.percentile(pk, 16, axis=1)
        prediction_summary_df[f'{belief_col}_prob_cat_upper_68_{cat}'] = np.percentile(pk, 84, axis=1)
    
    # --- Expected value draws df ---
    expected_value_draws_df = pd.DataFrame(
        ev,  # (n_cells, n_draws)
        columns=[f'draw_{i+1}' for i in range(n_draws)]
    )
    expected_value_draws_df.insert(0, 'cell_id', cell_id)
    
    # --- Category probability draws df ---
    prob_draw_cols = []
    prob_draw_blocks = []
    for k in range(K):
        cols_k = [f'prob_cat_{k+1}_draw_{i+1}' for i in range(n_draws)]
        prob_draw_cols.extend(cols_k)
        prob_draw_blocks.append(pr[:, :, k])  # (n_cells, n_draws)
    
    prob_category_draws_df = pd.DataFrame(
        np.concatenate(prob_draw_blocks, axis=1),
        columns=prob_draw_cols
    )
    prob_category_draws_df.insert(0, 'cell_id', cell_id)
    
    return prediction_summary_df, expected_value_draws_df, prob_category_draws_df, K, draw_indices

In [29]:
# weighted group means and summaries

def aggregate_draws_by_group(draws_df, group_col):
    """
    Aggregate expected value draws by a single group column with posterior weight draws.
    - For each weight draw, compute weighted aggregate of belief draws
    - Average the aggregated results across all weight draws
    """
    draw_cols = [c for c in draws_df.columns if c.startswith('draw_')]
    if not draw_cols:
        raise ValueError("No draw_* columns found in draws_df.")

    # Check for posterior weight draws (N_star.1, N_star.2, ..., N_star.104)
    n_star_cols = [c for c in draws_df.columns if c.startswith('N_star.')]
    n_star_draws = sorted([c for c in n_star_cols if re.match(r'N_star\.\d+$', c)])
    
    print(f"Using {len(n_star_draws)} posterior weight draws for aggregation")
    all_results = []
        
    weight_indices = np.random.choice(len(n_star_draws), size = len(draw_cols))
    selected_weights = [n_star_draws[idx] for idx in weight_indices]

    for draw_col, weight_col in zip(draw_cols, selected_weights):
        w = draws_df[weight_col].astype(float)
        weighted_draw = draws_df[draw_col] * w
        num = weighted_draw.groupby(draws_df[group_col]).sum() #the sum for the expected belief value in the demographic group
        den = w.groupby(draws_df[group_col]).sum() # the sum of weights in one demographic group
        group_result = (num/den).reset_index()
        group_result.columns = [group_col, draw_col]
        group_result.set_index(group_col, inplace=True)  # Set group_col as index for proper concatenation
        all_results.append(group_result)
    
    # Concatenate along axis=1 and reset index to get the group column back
    group_draws = pd.concat(all_results, axis=1).reset_index()

    return group_draws #(row = n_groups, columns = group_col + draw_cols)

def summarize_group_draws(group_draws_df, group_col):
    draw_cols = [c for c in group_draws_df.columns if c.startswith('draw_')]
    vals = group_draws_df[draw_cols]
    out = pd.DataFrame({group_col: group_draws_df[group_col]})
    out['pred_mean'] = vals.mean(axis=1)
    out['pred_median'] = vals.median(axis=1)
    out['pred_lower_95'] = vals.quantile(0.025, axis=1)
    out['pred_upper_95'] = vals.quantile(0.975, axis=1)
    out['pred_lower_90'] = vals.quantile(0.05, axis=1)
    out['pred_upper_90'] = vals.quantile(0.95, axis=1)
    out['pred_lower_80'] = vals.quantile(0.10, axis=1)
    out['pred_upper_80'] = vals.quantile(0.90, axis=1)
    out['pred_lower_68'] = vals.quantile(0.16, axis=1)
    out['pred_upper_68'] = vals.quantile(0.84, axis=1)
    out['pred_sd'] = vals.std(axis=1)
    return out # (row = n_groups, columns = group_col + 5 summary stats)

In [30]:
# compute group-level category probabilities from cell-level predictions
def compute_group_category_probabilities(pred_df, belief_col, group_col):
    """
    - For each weight draw, compute weighted group category probabilities
    - Average results across all 104 weight draws
    """
    cat_cols = [c for c in pred_df.columns if c.startswith(f"{belief_col}_prob_cat_mean_")]
    K = max(sorted(int(c.split('_')[-1]) for c in cat_cols)) #number of categories

    n_star_cols = [c for c in pred_df.columns if c.startswith('N_star.')]
    n_star_draws = sorted([c for c in n_star_cols if re.match(r'N_star\.\d+$', c)])
    
    # Randomly sample one weight for each category
    weight_indices = np.random.choice(len(n_star_draws), size=K)
    selected_weights = [n_star_draws[idx] for idx in weight_indices]
    
    # Compute weighted means per category using randomly selected weights
    out_rows = []
    for k in range(1, K+1):
        col_k = f"{belief_col}_prob_cat_mean_{k}"
        if col_k not in pred_df.columns:
            continue
        
        # Use randomly selected weight for this category
        w_col = selected_weights[k-1]
        w = pred_df[w_col].astype(float)
        
        # Compute weighted aggregation
        num_by_group = (pred_df[col_k] * w).groupby(pred_df[group_col]).sum()
        denom_by_group = w.groupby(pred_df[group_col]).sum()
        prob_by_group = (num_by_group / denom_by_group).reset_index()
        prob_by_group.columns = [group_col, 'prob_mean']
        prob_by_group['category'] = k
        out_rows.append(prob_by_group)
    
    if not out_rows:
        raise ValueError(f"No results generated during aggregation. Check that belief column '{belief_col}' and weight columns are present in pred_df.")
    
    final_result = pd.concat(out_rows, ignore_index=True)
    
    return final_result # (row = n_groups * K, columns = group_col, category, prob_mean)

In [31]:

# ----- Generate window-specific predictions from same posterior -----
import gc


time_window_mapping = (
    df[['time_window_start', 'time_window_start_idx']]
    .drop_duplicates()
    .sort_values('time_window_start_idx')
)

# Group mapping for consolidation
group_map = [
    ('state_simple', 'state'),
    ('state_idx', 'state_idx'),
    ('sex_idx', 'sex'),
    ('age_idx', 'age'),
    ('eth_idx', 'eth'),
    ('income_idx', 'income'),
    ('vote_idx', 'vote'),
    ('education_idx', 'education')
]

# columns needed only for downstream aggregation / weighting
n_star_cols = [c for c in strat_frame.columns if c.startswith('N_star.')]
n_star_draws = sorted([c for c in n_star_cols if re.match(r'N_star\.\d+$', c)])
merge_cols = ['state_simple', 'state_idx', 'sex_idx', 'age_idx', 'eth_idx', 'income_idx', 'vote_idx', 'education_idx']
# merge_cols = [c for c in merge_cols if c in strat_frame.columns]
broad_cols = ['age_broad', 'eth_broad', 'education_broad', 'income_broad', 'vote_broad', 'sex_broad']
meta_cols = ['N_star.median'] + merge_cols + n_star_draws + broad_cols

# Initialize storage for consolidated category probabilities across all windows
consolidated_cat_probs = {name: [] for _, name in group_map}

for i, row in enumerate(time_window_mapping.itertuples(index=False), start=1):
    
    window = row.time_window_start
    current_time_idx = int(row.time_window_start_idx)

    pred_time_idx_arr = np.full(N_pred, current_time_idx, dtype=int)

    stan_data_pred = dict(stan_data_base)
    stan_data_pred.update({
        "pred_time_idx": pred_time_idx_arr.tolist()
    })

    window_label = pd.Timestamp(window).strftime('%Y-%m-%d')
    model_dir = f"{base_dir}/{belief_col}_{window_label}_stan_output"
    os.makedirs(model_dir, exist_ok=True)

    print(f"[{i}/{len(time_window_mapping)}] Generating predictions for window={window_label} (time_idx={current_time_idx})")
    gq = model.generate_quantities(
        data=stan_data_pred,
        previous_fit=fit
    )

    prediction_summary_df, expected_value_draws_df, prob_category_draws_df, K, draw_indices = generate_cell_level_predictions_from_gq(
    gq,
    strat_frame,
    belief_col,
    n_keep_draws=250
    )

    # Free gq immediately after extracting draws
    del gq
    gc.collect()

    # stratification rows must exactly match generated cell-level rows
    n_cells_expected = len(strat_frame)
    if (
        len(prediction_summary_df) != n_cells_expected
        or len(expected_value_draws_df) != n_cells_expected
        or len(prob_category_draws_df) != n_cells_expected
    ):
        warning_msg = (
            "WARNING: Length mismatch detected between stratification frame and cell-level outputs. "
            "Stopping execution with no fallback."
        )
        print(warning_msg)
        raise ValueError(warning_msg)

    # Save compact cell-level outputs
    prediction_summary_df.to_csv(f"{model_dir}/cell_level_predictions.csv", index=False)

    # Save selected draw indices
    pd.Series(draw_indices, name='draw_index').to_csv(f"{model_dir}/selected_draw_indices.csv", index=False)

    # Build merge-ready metadata frame (stable key + original order)
    merge_meta = strat_frame.reset_index(drop=True).loc[:, meta_cols].copy()
    merge_meta.insert(0, 'cell_id', np.arange(len(merge_meta), dtype=int))

    # Temporary frames for weighted aggregation only
    draws_for_agg = expected_value_draws_df.merge(merge_meta, on='cell_id', how='left')
    pred_for_agg = prediction_summary_df.merge(merge_meta, on='cell_id', how='left')
    
    # Free draws df after merge — no longer needed raw
    del expected_value_draws_df, prob_category_draws_df, merge_meta
    gc.collect()

    # State-level aggregation and save (expected value draws)
    state_group_col = 'state_simple' if 'state_simple' in draws_for_agg.columns else ('state_idx' if 'state_idx' in draws_for_agg.columns else None)
    if state_group_col is not None:
        state_all_draws = aggregate_draws_by_group(draws_for_agg, state_group_col)
        state_preds = summarize_group_draws(state_all_draws, state_group_col)
        state_all_draws.to_csv(f"{model_dir}/group_state_all_draws.csv", index=False)
        state_preds.to_csv(f"{model_dir}/group_state_predictions.csv", index=False)
    else:
        print("Warning: no state column found; skipping state-level aggregation.")

    # Demographic group aggregations and save (expected value draws)
    demo_group_map = [
    ('sex_broad',       'sex'),
    ('age_broad',       'age'),
    ('eth_broad',       'eth'),
    ('income_broad',    'income'),
    ('vote_broad',      'vote'),
    ('education_broad', 'education'),
]
    for gcol, gname in demo_group_map:
        if gcol in draws_for_agg.columns:
            grp_draws = aggregate_draws_by_group(draws_for_agg, gcol)
            grp_preds = summarize_group_draws(grp_draws, gcol)
            grp_draws.to_csv(f"{model_dir}/group_{gname}_all_draws.csv", index=False)
            grp_preds.to_csv(f"{model_dir}/group_{gname}_predictions.csv", index=False)
        else:
            print(f"Note: column '{gcol}' not found; skipping group '{gname}'.")
    
    # Free draws_for_agg after all draw-based aggregations are done
    del draws_for_agg
    gc.collect()

    # Compute category probabilities for all groups and store for consolidation
    print(f"  Computing category probabilities ...")
    for gcol, gname in group_map:
        if gcol not in pred_for_agg.columns:
            continue
        try:
            grp_cat = compute_group_category_probabilities(
                pred_for_agg, belief_col, gcol
            )
            grp_cat['time_window_start'] = window
            consolidated_cat_probs[gname].append(grp_cat)
        except Exception as e:
            print(f"    Warning: failed to compute {gname} category probs: {e}")
    # Free remaining per-window frames
    del pred_for_agg, prediction_summary_df
    gc.collect()
    
    print(f"  Successfully generated all predictions for window {window}\n")    
print("All time windows processed!")

# Save consolidated category probability timeseries files
print("\nSaving consolidated category probability timeseries...")
for gname, parts in consolidated_cat_probs.items():
    if parts:
        df_all = pd.concat(parts, ignore_index=True)
        output_path = f"{base_dir}/group_{gname}_category_probs_timeseries.csv"
        df_all.to_csv(output_path, index=False)
        print(f" Saved {gname}: {df_all.shape[0]} rows, {df_all['time_window_start'].nunique()} windows")
    else:
        print(f" No data for {gname}")

print("\nPrediction and consolidation pipeline complete!")
# Cleanup: remove CmdStan output csv files 
for f in fit.runset.csv_files + fit.runset.stdout_files:
    if os.path.exists(f):
        os.remove(f)

del fit
gc.collect()

[1/26] Generating predictions for window=2023-11-05 (time_idx=1)


02:02:37 - cmdstanpy - INFO - Chain [1] start processing
02:02:37 - cmdstanpy - INFO - Chain [2] start processing
02:02:37 - cmdstanpy - INFO - Chain [3] start processing
02:02:37 - cmdstanpy - INFO - Chain [4] start processing
02:02:37 - cmdstanpy - INFO - Chain [5] start processing
02:02:37 - cmdstanpy - INFO - Chain [6] start processing
02:02:37 - cmdstanpy - INFO - Chain [7] start processing
02:02:37 - cmdstanpy - INFO - Chain [8] start processing
02:03:32 - cmdstanpy - INFO - Chain [7] done processing
02:03:32 - cmdstanpy - INFO - Chain [6] done processing
02:03:32 - cmdstanpy - INFO - Chain [1] done processing
02:03:32 - cmdstanpy - INFO - Chain [3] done processing
02:03:32 - cmdstanpy - INFO - Chain [8] done processing
02:03:32 - cmdstanpy - INFO - Chain [2] done processing
02:03:32 - cmdstanpy - INFO - Chain [5] done processing
02:03:32 - cmdstanpy - INFO - Chain [4] done processing


Subsampling draws: using 250 out of 256 posterior draws
Using 104 posterior weight draws for aggregation
Using 104 posterior weight draws for aggregation
Using 104 posterior weight draws for aggregation
Using 104 posterior weight draws for aggregation
Using 104 posterior weight draws for aggregation
Using 104 posterior weight draws for aggregation
Using 104 posterior weight draws for aggregation
  Computing category probabilities ...
  Successfully generated all predictions for window 2023-11-05 00:00:00+00:00

[2/26] Generating predictions for window=2023-11-19 (time_idx=2)


02:05:03 - cmdstanpy - INFO - Chain [1] start processing
02:05:03 - cmdstanpy - INFO - Chain [2] start processing
02:05:03 - cmdstanpy - INFO - Chain [3] start processing
02:05:03 - cmdstanpy - INFO - Chain [4] start processing
02:05:03 - cmdstanpy - INFO - Chain [5] start processing
02:05:03 - cmdstanpy - INFO - Chain [6] start processing
02:05:03 - cmdstanpy - INFO - Chain [7] start processing
02:05:03 - cmdstanpy - INFO - Chain [8] start processing
02:05:51 - cmdstanpy - INFO - Chain [8] done processing
02:05:51 - cmdstanpy - INFO - Chain [2] done processing
02:05:51 - cmdstanpy - INFO - Chain [5] done processing
02:05:51 - cmdstanpy - INFO - Chain [7] done processing
02:05:51 - cmdstanpy - INFO - Chain [6] done processing
02:05:51 - cmdstanpy - INFO - Chain [1] done processing
02:05:51 - cmdstanpy - INFO - Chain [3] done processing
02:05:51 - cmdstanpy - INFO - Chain [4] done processing


Subsampling draws: using 250 out of 256 posterior draws
Using 104 posterior weight draws for aggregation
Using 104 posterior weight draws for aggregation
Using 104 posterior weight draws for aggregation
Using 104 posterior weight draws for aggregation
Using 104 posterior weight draws for aggregation
Using 104 posterior weight draws for aggregation
Using 104 posterior weight draws for aggregation
  Computing category probabilities ...
  Successfully generated all predictions for window 2023-11-19 00:00:00+00:00

[3/26] Generating predictions for window=2023-12-03 (time_idx=3)


02:07:20 - cmdstanpy - INFO - Chain [1] start processing
02:07:20 - cmdstanpy - INFO - Chain [2] start processing
02:07:20 - cmdstanpy - INFO - Chain [3] start processing
02:07:20 - cmdstanpy - INFO - Chain [4] start processing
02:07:20 - cmdstanpy - INFO - Chain [5] start processing
02:07:20 - cmdstanpy - INFO - Chain [6] start processing
02:07:20 - cmdstanpy - INFO - Chain [7] start processing
02:07:20 - cmdstanpy - INFO - Chain [8] start processing
02:08:04 - cmdstanpy - INFO - Chain [2] done processing
02:08:04 - cmdstanpy - INFO - Chain [8] done processing
02:08:04 - cmdstanpy - INFO - Chain [4] done processing
02:08:04 - cmdstanpy - INFO - Chain [3] done processing
02:08:05 - cmdstanpy - INFO - Chain [7] done processing
02:08:05 - cmdstanpy - INFO - Chain [1] done processing
02:08:05 - cmdstanpy - INFO - Chain [6] done processing
02:08:05 - cmdstanpy - INFO - Chain [5] done processing


... (44 outputs left out)

Subsampling draws: using 250 out of 256 posterior draws
Using 104 posterior weight draws for aggregation
Using 104 posterior weight draws for aggregation
Using 104 posterior weight draws for aggregation
Using 104 posterior weight draws for aggregation
Using 104 posterior weight draws for aggregation
Using 104 posterior weight draws for aggregation
Using 104 posterior weight draws for aggregation
  Computing category probabilities ...
  Successfully generated all predictions for window 2024-10-06 00:00:00+00:00

[26/26] Generating predictions for window=2024-10-20 (time_idx=26)


02:56:35 - cmdstanpy - INFO - Chain [1] start processing
02:56:35 - cmdstanpy - INFO - Chain [2] start processing
02:56:35 - cmdstanpy - INFO - Chain [3] start processing
02:56:35 - cmdstanpy - INFO - Chain [4] start processing
02:56:35 - cmdstanpy - INFO - Chain [5] start processing
02:56:35 - cmdstanpy - INFO - Chain [6] start processing
02:56:35 - cmdstanpy - INFO - Chain [7] start processing
02:56:35 - cmdstanpy - INFO - Chain [8] start processing
02:57:19 - cmdstanpy - INFO - Chain [7] done processing
02:57:19 - cmdstanpy - INFO - Chain [4] done processing
02:57:19 - cmdstanpy - INFO - Chain [3] done processing
02:57:19 - cmdstanpy - INFO - Chain [2] done processing
02:57:19 - cmdstanpy - INFO - Chain [5] done processing
02:57:19 - cmdstanpy - INFO - Chain [1] done processing
02:57:19 - cmdstanpy - INFO - Chain [6] done processing
02:57:19 - cmdstanpy - INFO - Chain [8] done processing


Subsampling draws: using 250 out of 256 posterior draws
Using 104 posterior weight draws for aggregation
Using 104 posterior weight draws for aggregation
Using 104 posterior weight draws for aggregation
Using 104 posterior weight draws for aggregation
Using 104 posterior weight draws for aggregation
Using 104 posterior weight draws for aggregation
Using 104 posterior weight draws for aggregation
  Computing category probabilities ...
  Successfully generated all predictions for window 2024-10-20 00:00:00+00:00

All time windows processed!

Saving consolidated category probability timeseries...
 Saved state: 6630 rows, 26 windows
 Saved state_idx: 6630 rows, 26 windows
 Saved sex: 260 rows, 26 windows
 Saved age: 780 rows, 26 windows
 Saved eth: 780 rows, 26 windows
 Saved income: 650 rows, 26 windows
 Saved vote: 780 rows, 26 windows
 Saved education: 260 rows, 26 windows

Prediction and consolidation pipeline complete!


0